# Select metamodel goals in the AoI buffer ring

Workflow:

1. Auto-crop the **world-scale** hindcast catalogs and land polygons down to this
   AoI (once per case) into a small `hindcasts-<case>` folder, then load **AoI**,
   **land**, and all hindcast coordinates catalogs from it.
2. Build the **inland buffer ring**: `ring = (AoI - AoI.buffer(-d)) - land`.
3. Place **random spaced targets** inside the ring (open water), then number them **clockwise along the AoI boundary** (starting at the westernmost point).
4. For each target, snap to the **nearest node in every hindcast** catalog.
5. (Optional) Filter out goals that are too far from any hindcast node.
6. Compute **adaptive open-sea propagation sectors** from the nearest **AoI boundary** point.
7. (Optional) Manually override goal directions, re-plot, and then save JSON + GeoPackage.

**Direction convention** (`utils/directions.py`): all stored `angles` are **nautical**
wave-from directions, **clockwise from North** (`0°` = N, `90°` = E). Each goal has
`[left, right]` with `left < right` (clockwise sector boundaries). If the sector crosses
North, `left` is negative (e.g. `[-10, 50]` = 350°->50° clockwise). This matches WHACS
and SnapWave forcing; LHS samples `dir` in that range and normalizes with `% 360`.

**World master data**: `QGIS/HyWorld/hindcasts/` and `QGIS/HyWorld/land/` are
expected to hold the global hindcast node catalogs and a world coastline
(e.g. GSHHS). They're too large for git (`QGIS/` is in `.gitignore`) and too
slow to load directly in Python, so they live only on disk, however you
obtain them. This notebook reads only the small AoI-sized crop it needs from
them (`prepare_case_hindcasts`, below); the cropped `hindcasts-<case>/*.gpkg`
files it writes are small enough to drag-and-drop straight into QGIS.

In [ ]:
import json

import geopandas as gpd
import pandas as pd
from utils.config import CASE_DIR, PATHS, resolve
from utils.geometry import sectors as hc
from utils.viz import case_plots as plot

## Parameters

In [ ]:
buffer_distance_m = 5_000.0
min_spacing_m = 40_000.0

# Optional cap: drop goals if any hindcast snap distance exceeds this threshold.
# Set to None to disable filtering.
max_hindcast_distance_m = 60_000.0

sector_margin_deg = 5.0
random_seed = 42

geo_crs = "EPSG:4326"
work_crs = "EPSG:32702"  # TODO: UTM CRS for your region

# World-scale masters (not in git -- see the intro cell). Cropped once per case
# into hindcasts_dir by prepare_case_hindcasts(), below.
world_hindcasts_dir = CASE_DIR.parent / "QGIS/HyWorld/hindcasts"
world_land_path = CASE_DIR.parent / "QGIS/HyWorld/land/GSHHS_f_L1.shp"

hindcasts_dir = CASE_DIR.parent / "QGIS/HyWorld" / f"hindcasts-{CASE_DIR.name.lower()}"
land_path = hindcasts_dir / "land.gpkg"

# Crop padding around the AoI bbox -- must cover max_hindcast_distance_m.
hindcasts_buffer_km = max(150.0, (max_hindcast_distance_m or 0) / 1_000 * 1.2)

from utils.config import SIMULATIONS

goals_out_path = resolve(SIMULATIONS["goals_file"])
goals_gpkg_path = goals_out_path.with_suffix(".gpkg")

print(f"AoI       : {resolve(PATHS['aoi_file'])}")
print(f"hindcasts : {hindcasts_dir}")
print(f"land      : {land_path}")
print(f"json      : {goals_out_path}")
print(f"gpkg      : {goals_gpkg_path}")
print(f"min dist  : {min_spacing_m} m")
print(f"max dist  : {max_hindcast_distance_m} m")

## Prepare per-case hindcasts (crop world masters to this AoI)

No-op if `hindcasts_dir` already exists -- pass `overwrite=True` (or delete the
folder) to rebuild, e.g. after moving the AoI or widening
`max_hindcast_distance_m`.

In [ ]:
aoi = gpd.read_file(resolve(PATHS["aoi_file"])).to_crs(geo_crs)

hc.prepare_case_hindcasts(
    aoi,
    out_dir=hindcasts_dir,
    world_hindcasts_dir=world_hindcasts_dir,
    world_land_path=world_land_path,
    buffer_km=hindcasts_buffer_km,
)
print(f"hindcasts ready: {hindcasts_dir}")

## Load inputs

In [ ]:
land = hc.load_land(land_path)
catalogs = hc.load_catalogs(hindcasts_dir)

print(f"AoI catalogs: {list(catalogs)}")
for name, gdf in catalogs.items():
    print(f"  {name:6s}: {len(gdf):4d} points")

## Buffer ring and random targets

In [ ]:
ring_poly, ring_gdf = hc.build_ring(aoi, land, buffer_distance_m, work_crs)
targets = hc.random_targets_in_ring(
    ring_poly,
    land,
    min_spacing_m=min_spacing_m,
    work_crs=work_crs,
    random_seed=random_seed,
)
targets = hc.sort_targets_along_aoi(
    targets,
    aoi,
    work_crs=work_crs,
    clockwise=True,  # False = CCW; None = shapefile vertex order
    start_at="west",  # goal 1 starts at the westernmost AoI point
)

print(
    f"ring area : {gpd.GeoDataFrame(geometry=[ring_poly], crs=geo_crs).to_crs(work_crs).geometry.area.iloc[0] / 1e6:,.1f} km²"
)
print(f"targets   : {len(targets)}  (min spacing = {min_spacing_m / 1000:g} km)")
targets

## Build goals dict (target, angles, hindcast snaps)

`angles` = nautical clockwise `[left, right]` per goal (see notebook intro).

In [ ]:
sectors = hc.AoIRingSectors.from_aoi(aoi, work_crs)
goals = hc.build_goals_dict(
    targets,
    catalogs,
    sectors,
    work_crs=work_crs,
    margin_deg=sector_margin_deg,
)

if max_hindcast_distance_m is not None:
    goals = {
        goal_id: goal
        for goal_id, goal in goals.items()
        if max(v["distance_m"] for v in goal["hindcast"].values())
        <= max_hindcast_distance_m
    }
    keep_ids = set(goals)
    targets = targets[targets["goal_id"].astype(str).isin(keep_ids)].copy()

boundary_snaps = hc.boundary_snaps(targets, sectors, work_crs=work_crs)

summary_df = pd.DataFrame(
    [
        {
            "goal": k,
            "target_lon": v["target"]["longitude"],
            "target_lat": v["target"]["latitude"],
            "angles": v["angles"],
            "max_dist_m": max(h["distance_m"] for h in v["hindcast"].values()),
            **{f"{h}_fid": v["hindcast"][h]["fid"] for h in v["hindcast"]},
            **{f"{h}_dist_m": v["hindcast"][h]["distance_m"] for h in v["hindcast"]},
        }
        for k, v in goals.items()
    ]
)

print(f"goals kept: {len(goals)}")
summary_df

## Optional direction overrides (edit and re-run)

You can manually adjust directions before plotting/saving.
Use nautical clockwise `[left, right]` with `left < right`.

Example:

```python
direction_overrides = {
    "1": [-20, 60],
    "3": [95, 180],
}
```

In [ ]:
# Optionally edit this dictionary and rerun this cell.
direction_overrides: dict[str, list[float]] = {
    "4": [-90, 140],
    "5": [-90, 135],
    "6": [-90, 135],
    "15": [80, 300],
}

goals_final = json.loads(json.dumps(goals))
for goal_id, angles in direction_overrides.items():
    if goal_id not in goals_final:
        raise KeyError(f"Unknown goal_id in direction_overrides: {goal_id}")
    if len(angles) != 2 or angles[0] >= angles[1]:
        raise ValueError(
            f"{goal_id}: angles must be [left, right] with left < right, got {angles}"
        )
    goals_final[goal_id]["angles"] = [float(angles[0]), float(angles[1])]

fig, ax = plot.plot_base_map(
    title="Goals",
    show_topobathy=True,
    show_satellite=True,
)
plot.plot_sectors(targets, goals_final, boundary_snaps, ax)
targets.plot(ax=ax, color="black", markersize=40, marker="x", label="target")

plot.plot_points(
    hc.hindcast_snaps_gdf(goals_final),
    ax,
    color_col="catalog",
    label_col="goal_id",
    color_map=plot.HINDCAST_COLORS,
)

minx, miny, maxx, maxy = aoi.total_bounds
pad = 0.4
ax.set_xlim(minx - pad, maxx + pad)
ax.set_ylim(miny - pad, maxy + pad)
ax.set_aspect("equal", adjustable="box")

fig.tight_layout()

## Save goals JSON and QGIS GeoPackage

After saving, update the case JSON with your selected goals JSON path:

```json
"simulations": {
    "goals_file": "inputs/goals/<Case>_Goals.json"
}
```

In [ ]:
goals_to_save = goals_final if "goals_final" in locals() else goals

goals_out_path.parent.mkdir(parents=True, exist_ok=True)

with goals_out_path.open("w", encoding="utf-8") as f:
    json.dump({"goals": goals_to_save}, f, indent=4)
    f.write("\n")

hc.save_goals_gpkg(
    goals_to_save, targets, boundary_snaps, goals_gpkg_path, ring=ring_gdf
)

print(f"Wrote {len(goals_to_save)} goal(s) to:")
print(f"  {goals_out_path}")
print(f"  {goals_gpkg_path}")
print(
    "\nGPKG layers: targets, boundary_snaps, sectors, links, buffer_ring, "
    "codec, era5, whacs"
)
print(json.dumps({"goals": goals_to_save}, indent=4))